# Approved H1/H2 pilot — v006

User approved only H1's controlled duration/schedule tests and H2's lung-setting
output heads. This notebook waits for the prior C/final pipeline to finish via
its external runner, and independently enforces its final-manifest gate.
All modeling uses the approved data, original grouped A split, native inspiratory
MAE, training-only preprocessing and best held-out checkpoints.

H1 duration extends the exact saved 200-epoch trajectory with 100 floor-rate
epochs; a documented, tested checkpoint migration preserves Adam/RNG/data order
and changes only the declared budget/rate suffix. H1 schedule starts fresh at
300 epochs with slower cosine cooling. H2 changes only the output-head flag at
200 epochs. The three candidates are also compared by replacing the compact
member of the existing .5/.5 ensemble; the deep member and weights stay fixed.

Default review performs no fits. `PA3_RUN_PILOT_V006=1` enables the approved
screen; `PA3_RESUME_PILOT_V006=1` permits compatible saved states. Grouped
confirmation and any final refits remain separate, conditional on the results.


In [1]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import gc,copy,json,hashlib,time,uuid,fcntl
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import pandas as pd
RUN=os.environ.get('PA3_RUN_PILOT_V006','0')=='1'
RESUME=os.environ.get('PA3_RESUME_PILOT_V006','0')=='1'
PROJECT=Path.cwd().resolve()
OUT=PROJECT/'outputs/approved_pilots_v006'
BASE=PROJECT/'outputs/improvements_v005/sequence'
FINAL_GATE=PROJECT/'outputs/improvements_v005/confirmation_round1/final_all/final_manifest.json'
SEED=20261102
PHASE_LIMIT='approved_H1_H2_only'
PLAN=dict(version='approved_pilot_v006',approval='User approved H1 and H2 after current confirmation/final fits',trials=['H1_duration300','H1_slowcosine300','H2_RC_heads200'],feature_mode='combined',baseline='capacity_bilstm128',training_breaths=61114,validation_breaths=6791,metric='native inspiratory MAE',same_split_seed_loss_scaling=True,ensemble_check='replace compact member of existing equal-weight pair; deep member and .5/.5 weights fixed',new_model_seeds=False,H3_H4_in_scope=False,new_C_or_final_fits_in_this_notebook=False,submission=False)
def sha(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
 return h.hexdigest()
def signature(value):return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()
def code_hash(path):
 n=json.loads(Path(path).read_text());return hashlib.sha256('\n'.join(''.join(c['source']) for c in n['cells'] if c['cell_type']=='code').encode()).hexdigest()
def write_json(path,value):
 path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.tmp-'+uuid.uuid4().hex);tmp.write_text(json.dumps(value,indent=2,allow_nan=False)+'\n');os.replace(tmp,path)
print('Approved H1/H2 pilot:', 'execute' if RUN else 'review', '|',OUT)


Approved H1/H2 pilot: execute | /home/v2gao/dsc102/pa3/gym-workspace/ventilator-pressure-prediction/outputs/approved_pilots_v006


In [2]:
if RUN:
    import hashlib
    import json
    from pathlib import Path

    import numpy as np
    import pandas as pd

    STEPS = 80
    V005_FEATURE_MODES = ('causal20', 'offline', 'dynamics', 'combined')
    V005_BASE_FEATURE_NAMES = (
        'R', 'C', 'time_step', 'u_in', 'u_out', 'step', 'dt',
        'control_sum', 'control_mean', 'control_integral',
        'u_in_lag_1', 'u_in_change_1', 'u_in_lag_2', 'u_in_change_2',
        'u_in_lag_4', 'u_in_change_4', 'u_in_lag_8', 'u_in_change_8',
        'u_in_mean_4', 'u_in_mean_8',
    )
    V005_OFFLINE_FEATURE_NAMES = (
        'u_in_lead_1', 'u_in_lead_2', 'u_in_lead_4', 'u_in_lead_8',
        'u_in_centered_mean_4', 'u_in_centered_mean_8',
        'inspiration_duration', 'inspiration_time_fraction',
        'time_to_u_out_switch', 'inspiration_u_in_peak',
        'inspiration_u_in_peak_time', 'inspiration_u_in_mean',
        'inspiration_u_in_std', 'inspiration_u_in_integral',
    )
    V005_EWM_TIMESCALES_SECONDS = (.05, .1, .2, .5, 1., 2.)
    V005_DYNAMICS_FEATURE_NAMES = (
        'rc_tau_seconds', 'time_over_rc_tau', 'u_in_times_R', 'u_in_over_R',
        'du_in_dt', 'control_trapezoid_integral',
        'control_trapezoid_over_compliance_L',
        'control_ewm_tau_0_05', 'control_ewm_tau_0_1',
        'control_ewm_tau_0_2', 'control_ewm_tau_0_5',
        'control_ewm_tau_1', 'control_ewm_tau_2',
    )

In [3]:
if RUN:
    def features(frame: pd.DataFrame) -> tuple[np.ndarray, list[str], np.ndarray]:
        """Return features in original row order, after verifying 80-row blocks."""
        if len(frame) % STEPS:
            raise ValueError("Unexpected breath length; do not reshape this file")
        blocks = frame["breath_id"].to_numpy().reshape(-1, STEPS)
        breath_ids = blocks[:, 0]
        if not np.all(blocks == breath_ids[:, None]):
            raise ValueError("Breaths are not contiguous 80-row blocks")
        if len(np.unique(breath_ids)) != len(breath_ids):
            raise ValueError("Repeated breath blocks")
        if not frame["id"].is_unique:
            raise ValueError("Duplicate row IDs")
        u = frame["u_in"].to_numpy().reshape(-1, STEPS)
        t = frame["time_step"].to_numpy().reshape(-1, STEPS)
        if not np.all(np.diff(t, axis=1) > 0):
            raise ValueError("Timestamps must increase inside each breath")
        for column in ("R", "C"):
            values = frame[column].to_numpy().reshape(-1, STEPS)
            if not np.all(values == values[:, :1]):
                raise ValueError(f"{column} changes within a breath")

        columns = {}
        for column in ("R", "C", "time_step", "u_in", "u_out"):
            columns[column] = frame[column].to_numpy(dtype=np.float32)
        step = np.arange(STEPS)
        dt = np.diff(t, axis=1, prepend=t[:, :1])
        cumulative = np.cumsum(u, axis=1, dtype=np.float32)
        columns["step"] = np.broadcast_to(step, u.shape).ravel()
        columns["dt"] = dt.ravel()
        columns["control_sum"] = cumulative.ravel()
        columns["control_mean"] = (cumulative / (step + 1)).ravel()
        # Valve opening is not measured flow: this is a history proxy, not volume.
        columns["control_integral"] = np.cumsum(u * dt, axis=1).ravel()
        for lag in (1, 2, 4, 8):
            shifted = np.empty_like(u)
            shifted[:, :lag] = u[:, :1]
            shifted[:, lag:] = u[:, :-lag]
            columns[f"u_in_lag_{lag}"] = shifted.ravel()
            columns[f"u_in_change_{lag}"] = (u - shifted).ravel()
        padded_sum = np.concatenate([np.zeros((len(u), 1), dtype=np.float32), cumulative], axis=1)
        for window in (4, 8):
            begin = np.maximum(0, step + 1 - window)
            columns[f"u_in_mean_{window}"] = (
                (padded_sum[:, step + 1] - padded_sum[:, begin]) / (step + 1 - begin)
            ).ravel()
        names = list(columns)
        matrix = np.column_stack(list(columns.values())).astype(np.float32, copy=False)
        if not np.isfinite(matrix).all() or "pressure" in names:
            raise ValueError("Invalid features")
        return matrix, names, breath_ids

In [4]:
if RUN:
    # Keep the legacy flat API available when importing these cells into a notebook.
    features_causal20 = features


    def _validate_feature_frame_v005(frame):
        """Validate known inputs only. Pressure and unrelated columns are never read."""
        if not isinstance(frame, pd.DataFrame):
            raise TypeError('Expected a pandas DataFrame')
        required = ('id', 'breath_id', 'R', 'C', 'time_step', 'u_in', 'u_out')
        if not frame.columns.is_unique:
            raise ValueError('Duplicate column names')
        missing = set(required) - set(frame.columns)
        if missing:
            raise ValueError(f'Missing required inputs: {sorted(missing)}')
        if not len(frame) or len(frame) % STEPS:
            raise ValueError('Expected nonempty contiguous 80-row breaths')
        for column in required:
            if not pd.api.types.is_numeric_dtype(frame[column]):
                raise ValueError(f'{column} must be numeric')
            values = frame[column].to_numpy()
            if not np.isfinite(values).all():
                raise ValueError(f'{column} must be finite')
        for column in ('id', 'breath_id'):
            values = frame[column].to_numpy()
            if not np.all(values == np.floor(values)):
                raise ValueError(f'{column} must contain integer IDs')
        if not frame['id'].is_unique:
            raise ValueError('Duplicate row IDs')
        blocks = frame['breath_id'].to_numpy().reshape(-1, STEPS)
        breath_ids = blocks[:, 0]
        if not np.all(blocks == breath_ids[:, None]):
            raise ValueError('Breaths are not contiguous 80-row blocks')
        if len(np.unique(breath_ids)) != len(breath_ids):
            raise ValueError('Repeated breath blocks')
        t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        if not np.all(np.diff(t, axis=1) > 0):
            raise ValueError('Timestamps must increase inside each breath')
        for column, levels in (('R', (5, 20, 50)), ('C', (10, 20, 50))):
            values = frame[column].to_numpy().reshape(-1, STEPS)
            if not np.all(values == values[:, :1]):
                raise ValueError(f'{column} changes within a breath')
            if not np.isin(values, levels).all():
                raise ValueError(f'Unexpected {column} level')
        out = frame['u_out'].to_numpy().reshape(-1, STEPS)
        if not np.isin(out, (0, 1)).all():
            raise ValueError('u_out must be binary')
        # The approved data has a single inspiration followed by expiration. A fully
        # observed inspiration without an expiration sample is supported as well.
        if np.any(out[:, 0] != 0) or np.any(np.diff(out.astype(np.int8), axis=1) < 0):
            raise ValueError('Expected initial inspiration followed by at most one valve opening')
        return breath_ids


    def feature_names_v005(feature_mode='causal20'):
        if feature_mode not in V005_FEATURE_MODES:
            raise ValueError(f'Unknown feature_mode: {feature_mode!r}')
        names = list(V005_BASE_FEATURE_NAMES)
        if feature_mode in ('offline', 'combined'):
            names.extend(V005_OFFLINE_FEATURE_NAMES)
        if feature_mode in ('dynamics', 'combined'):
            names.extend(V005_DYNAMICS_FEATURE_NAMES)
        return names


    def _offline_feature_block_v005(frame):
        """Known whole-breath input context; no target values or statistics."""
        u = frame['u_in'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        out = frame['u_out'].to_numpy().reshape(-1, STEPS)
        inspiration = out == 0
        n_breaths = len(u)
        columns = []
        for lead in (1, 2, 4, 8):
            shifted = np.empty_like(u)
            shifted[:, :-lead] = u[:, lead:]
            shifted[:, -lead:] = u[:, -1:]
            columns.append(shifted)
        prefix = np.concatenate((np.zeros((n_breaths, 1)), np.cumsum(u, axis=1)), axis=1)
        steps = np.arange(STEPS)
        for window in (4, 8):
            # Even centered windows have window//2 prior and window//2-1 future
            # neighbours, matching pandas rolling(center=True, min_periods=1).
            begin = np.maximum(0, steps - window//2)
            end = np.minimum(STEPS, steps + (window+1)//2)
            columns.append((prefix[:, end] - prefix[:, begin]) / (end-begin))
        has_expiration = np.any(out == 1, axis=1)
        first_expiration = np.argmax(out == 1, axis=1)
        switch_index = np.where(has_expiration, first_expiration, STEPS-1)
        switch_time = t[np.arange(n_breaths), switch_index]
        duration = switch_time - t[:, 0]
        # With strictly increasing 80-step timestamps and initial inspiration,
        # duration is positive, including the no-expiration boundary case.
        fraction = np.clip((t-t[:, :1])/duration[:, None], 0., 1.)
        until_switch = np.maximum(switch_time[:, None]-t, 0.)
        masked = np.where(inspiration, u, -np.inf)
        peak_index = np.argmax(masked, axis=1)  # first peak resolves a known-input tie
        peak = u[np.arange(n_breaths), peak_index]
        peak_time = t[np.arange(n_breaths), peak_index] - t[:, 0]
        count = inspiration.sum(axis=1)
        mean = np.sum(np.where(inspiration, u, 0.), axis=1)/count
        variance = np.sum(np.where(inspiration, (u-mean[:, None])**2, 0.), axis=1)/count
        std = np.sqrt(np.maximum(variance, 0.))
        # Integrate between inspiratory observations only; do not infer unobserved
        # flow or include the interval crossing into expiration.
        both_inspiration = inspiration[:, 1:] & inspiration[:, :-1]
        integral = np.sum(np.where(both_inspiration,
            .5*(u[:, 1:]+u[:, :-1])*np.diff(t, axis=1), 0.), axis=1)
        def broadcast(values):
            return np.broadcast_to(values[:, None], u.shape)
        columns.extend((broadcast(duration), fraction, until_switch, broadcast(peak),
            broadcast(peak_time), broadcast(mean), broadcast(std), broadcast(integral)))
        return np.stack(columns, axis=-1).astype(np.float32)


    def _dynamics_feature_block_v005(frame):
        """Causal timing and control proxies; valve opening is not measured flow."""
        u = frame['u_in'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        r = frame['R'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        c_litres = frame['C'].to_numpy(dtype=np.float64).reshape(-1, STEPS)/1000.
        dt = np.diff(t, axis=1, prepend=t[:, :1])
        tau = r*c_litres  # R in cmH2O/(L/s), C converted from mL/cmH2O
        derivative = np.zeros_like(u)
        derivative[:, 1:] = np.diff(u, axis=1)/dt[:, 1:]
        trapezoid = np.zeros_like(u)
        trapezoid[:, 1:] = np.cumsum(.5*(u[:, 1:]+u[:, :-1])*dt[:, 1:], axis=1)
        # These interactions can help the learner represent R/C sensitivity. They
        # are not asserted to have physical pressure/flow/volume units because u_in
        # is a valve-opening percentage, not observed flow or driving pressure.
        columns = [tau, (t-t[:, :1])/tau, u*r, u/r, derivative,
                   trapezoid, trapezoid/c_litres]
        scales = np.asarray(V005_EWM_TIMESCALES_SECONDS, dtype=np.float64)
        histories = np.empty((*u.shape, len(scales)), dtype=np.float32)
        state = np.broadcast_to(u[:, :1], (len(u), len(scales))).copy()
        histories[:, 0, :] = state
        # One loop over 80 timesteps; all breaths and timescales run together.
        # Irregular-time EWMA includes the control observed at the current step.
        for step in range(1, STEPS):
            alpha = np.exp(-dt[:, step, None]/scales[None, :])
            state = alpha*state + (1.-alpha)*u[:, step, None]
            histories[:, step, :] = state
        fixed = np.stack(columns, axis=-1).astype(np.float32)
        return np.concatenate((fixed, histories), axis=-1)


    def make_sequence_features(frame, feature_mode='causal20'):
        """Return (X[breaths,80,F] float32, names, breath_ids), in input order.

        causal20: exact legacy features; offline: legacy + known future/context;
        dynamics: legacy + causal dynamics proxies; combined: all three blocks.
        No scaling, fitting, target reads, IDs as features, or frame mutation occur.
        """
        names = feature_names_v005(feature_mode)
        breath_ids = _validate_feature_frame_v005(frame)
        base_flat, base_names, old_ids = features_causal20(frame)
        if base_names != list(V005_BASE_FEATURE_NAMES) or not np.array_equal(breath_ids, old_ids):
            raise ValueError('Legacy feature schema changed')
        base = base_flat.reshape(-1, STEPS, len(base_names))
        if feature_mode == 'causal20':
            return base, names, breath_ids.copy()
        result = np.empty((len(breath_ids), STEPS, len(names)), dtype=np.float32)
        result[..., :len(base_names)] = base
        position = len(base_names)
        if feature_mode in ('offline', 'combined'):
            block = _offline_feature_block_v005(frame)
            result[..., position:position+block.shape[-1]] = block
            position += block.shape[-1]
        if feature_mode in ('dynamics', 'combined'):
            block = _dynamics_feature_block_v005(frame)
            result[..., position:position+block.shape[-1]] = block
            position += block.shape[-1]
        if position != len(names) or not np.isfinite(result).all():
            raise ValueError('Invalid feature output')
        return result, names, breath_ids.copy()

In [5]:
if RUN:
    # Imports and model definitions
    """Reusable v005 components. Defining these cells performs no data reads or fits."""
    import os
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    import copy
    import hashlib
    import json
    import math
    import random
    import time
    import uuid
    from pathlib import Path
    import numpy as np
    import torch
    from torch import nn
    from torch.nn import functional as F
    from torch.utils.data import DataLoader, TensorDataset

    V5_STEPS = 80
    V5_R_VALUES = (5, 20, 50)
    V5_C_VALUES = (10, 20, 50)
    V5_MODEL_DEFAULTS = dict(family='gru', hidden=128, layers=2, dropout=.15,
        bidirectional=False, joint_embedding_dim=0, rc_heads=False, residual=False,
        head_width=32, widths=None, output_concat=False)
    V5_TRAIN_DEFAULTS = dict(lr=.0015, batch_size=256, max_epochs=150, patience=0,
        min_delta=1e-6, max_seconds=None, min_lr=1e-5, scheduler='plateau',
        scheduler_factor=.5, scheduler_patience=6, scheduler_threshold=1e-4,
        warmup_epochs=0, prediction_batch_size=512, gradient_clip=1.,
        optimizer='Adam', weight_decay=0., loss='l1', native_beta=.1,
        scaler_phase='all', residual_baseline=False, frozen_lr_schedule=None)


    def canonical_model_spec(model_spec):
        supplied = copy.deepcopy(dict(model_spec))
        unknown = set(supplied)-set(V5_MODEL_DEFAULTS)
        if unknown:
            raise ValueError(f'Unknown model fields: {sorted(unknown)}')
        spec = dict(V5_MODEL_DEFAULTS, **supplied)
        spec['family'] = str(spec['family']).lower()
        if spec['family'] not in ('gru', 'lstm'):
            raise ValueError('family must be gru or lstm')
        for key in ('hidden', 'layers', 'head_width'):
            spec[key] = int(spec[key])
            if spec[key] < 1:
                raise ValueError(f'{key} must be positive')
        spec['joint_embedding_dim'] = int(spec['joint_embedding_dim'])
        if spec['joint_embedding_dim'] < 0:
            raise ValueError('joint_embedding_dim must be nonnegative')
        spec['dropout'] = float(spec['dropout'])
        if not 0 <= spec['dropout'] < 1:
            raise ValueError('dropout must be in [0,1)')
        for key in ('bidirectional', 'rc_heads', 'residual', 'output_concat'):
            spec[key] = bool(spec[key])
        if spec['widths'] is not None:
            spec['widths'] = [int(v) for v in spec['widths']]
            if len(spec['widths']) != spec['layers'] or min(spec['widths']) < 1:
                raise ValueError('widths must contain one positive hidden width per layer')
        return spec


    def seed_sequence(seed):
        random.seed(int(seed))
        np.random.seed(int(seed))
        torch.manual_seed(int(seed))
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(int(seed))
        torch.set_num_threads(4)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True)
        if hasattr(torch.backends.cuda.matmul, 'allow_tf32'):
            torch.backends.cuda.matmul.allow_tf32 = False
        if hasattr(torch.backends.cudnn, 'allow_tf32'):
            torch.backends.cudnn.allow_tf32 = False


    def _sequence_device(device):
        result = torch.device(device)
        if result.type == 'cuda' and not torch.cuda.is_available():
            raise RuntimeError('CUDA requested but unavailable')
        return result


    class V5PressureRNN(nn.Module):
        """Independent complete breaths; no hidden or cell state persists across calls.

        Transformed inputs contain F standardized numeric features, followed by an
        internal joint R/C code. Numeric R/C always remain among those F features.
        Bidirectional variants intentionally use future inputs within the same breath.
        """
        def __init__(self, model_spec, n_features):
            super().__init__()
            self.spec = spec = canonical_model_spec(model_spec)
            self.n_features = int(n_features)
            if self.n_features < 2:
                raise ValueError('At least the physical R and C channels are required')
            embedding = spec['joint_embedding_dim']
            self.joint_embedding = nn.Embedding(9, embedding) if embedding else None
            input_width = self.n_features + embedding
            directions = 2 if spec['bidirectional'] else 1
            rnn_class = nn.GRU if spec['family'] == 'gru' else nn.LSTM
            # Native multi-layer cuDNN recurrent dropout keeps opaque internal RNG state.
            # Single-layer recurrent modules plus explicit Torch dropout preserve the
            # plain stacked model while making every stochastic state serializable.
            self.explicit_stack = bool(spec['widths'] is not None or spec['residual'] or
                spec['output_concat'] or (spec['layers'] > 1 and spec['dropout'] > 0))
            if not self.explicit_stack:
                self.rnn = rnn_class(input_width, spec['hidden'], num_layers=spec['layers'],
                    batch_first=True, bidirectional=spec['bidirectional'],
                    dropout=spec['dropout'] if spec['layers'] > 1 else 0.)
                head_input = spec['hidden'] * directions
            else:
                widths = spec['widths'] or [spec['hidden']]*spec['layers']
                self.recurrent_layers = nn.ModuleList()
                self.skip_projections = nn.ModuleList()
                self.between_layers = nn.Dropout(spec['dropout'])
                outputs = []
                for width in widths:
                    output_width = width*directions
                    self.recurrent_layers.append(rnn_class(input_width, width, num_layers=1,
                        batch_first=True, bidirectional=spec['bidirectional']))
                    self.skip_projections.append(nn.Linear(input_width, output_width)
                        if spec['residual'] else nn.Identity())
                    outputs.append(output_width)
                    input_width = output_width
                head_input = sum(outputs) if spec['output_concat'] else outputs[-1]
            self.head = nn.Sequential(nn.Linear(head_input, spec['head_width']), nn.SiLU(),
                nn.Linear(spec['head_width'], 9 if spec['rc_heads'] else 1))

        def forward(self, x):
            if x.ndim != 3 or x.shape[-1] != self.n_features+1:
                raise ValueError('Expected standardized numeric features plus internal R/C code')
            joint = x[..., -1].long()
            sequence = x[..., :self.n_features]
            if self.joint_embedding is not None:
                sequence = torch.cat((sequence, self.joint_embedding(joint)), dim=-1)
            if not self.explicit_stack:
                sequence, _ = self.rnn(sequence)
            else:
                outputs = []
                for i, layer in enumerate(self.recurrent_layers):
                    previous = sequence
                    sequence, _ = layer(previous)
                    if self.spec['residual']:
                        sequence = sequence + self.skip_projections[i](previous)
                    outputs.append(sequence)
                    if i+1 < len(self.recurrent_layers):
                        sequence = self.between_layers(sequence)
                if self.spec['output_concat']:
                    sequence = torch.cat(outputs, dim=-1)
            prediction = self.head(sequence)
            if self.spec['rc_heads']:
                return prediction.gather(-1, joint.unsqueeze(-1)).squeeze(-1)
            return prediction.squeeze(-1)


    def build_model(model_spec, n_features):
        return V5PressureRNN(model_spec, n_features)


    def model_parameter_count(model_spec, n_features):
        return int(sum(p.numel() for p in build_model(model_spec, n_features).parameters()))

In [6]:
if RUN:
    # Training-only preprocessing and native prediction
    def _raw_x(x):
        raw = np.asarray(x, dtype=np.float32)
        if raw.ndim != 3 or raw.shape[1] != V5_STEPS or raw.shape[2] < 2 or not len(raw):
            raise ValueError('Expected nonempty X[breaths,80,F], F >= 2')
        if not np.isfinite(raw).all():
            raise ValueError('Nonfinite feature')
        for column, values in ((0, V5_R_VALUES), (1, V5_C_VALUES)):
            if not np.isin(raw[..., column], values).all():
                raise ValueError('Unknown physical R/C value')
            if not np.all(raw[..., column] == raw[:, :1, column]):
                raise ValueError('R/C must be constant inside each breath')
        return raw


    def _training_arrays(x, y, u_out):
        raw = _raw_x(x)
        target, out = np.asarray(y, dtype=np.float32), np.asarray(u_out)
        if target.shape != raw.shape[:2] or out.shape != target.shape:
            raise ValueError('Feature/target/mask shape mismatch')
        if not np.isfinite(target).all() or not np.isin(out, [0, 1]).all() or not (out == 0).any():
            raise ValueError('Finite targets and a nonempty binary inspiration mask are required')
        return raw, target, out


    def _joint_codes(raw):
        r = np.searchsorted(np.asarray(V5_R_VALUES), raw[:, 0, 0])
        c = np.searchsorted(np.asarray(V5_C_VALUES), raw[:, 0, 1])
        return (3*r+c).astype(np.int64)


    def fit_preprocessor(x_train, y_train, u_out_train, model_spec, feature_names=None,
                         scaler_phase='all', residual_baseline=False):
        """All fitted statistics, including optional pressure profiles, use training only."""
        raw, y, out = _training_arrays(x_train, y_train, u_out_train)
        canonical_model_spec(model_spec)
        n_features = raw.shape[-1]
        names = list(feature_names) if feature_names is not None else None
        if names is not None and (len(names) != n_features or len(set(names)) != n_features):
            raise ValueError('feature_names must uniquely name every raw input channel')
        if scaler_phase not in ('all', 'inspiration'):
            raise ValueError('scaler_phase must be all or inspiration')
        mean = raw.mean(axis=(0, 1), dtype=np.float64).astype(np.float32)
        std = raw.std(axis=(0, 1), dtype=np.float64).astype(np.float32)
        phase_columns = []
        mask = out == 0
        if scaler_phase == 'inspiration':
            if names is None:
                raise ValueError('Inspiration-only scaling requires explicit feature_names')
            excluded = {'R', 'C', 'u_out', 'time_step', 'step'}
            phase_columns = [j for j, name in enumerate(names) if j >= 2 and name not in excluded]
            scored_x = raw[mask]
            phase_mean = scored_x.mean(axis=0, dtype=np.float64)
            phase_std = scored_x.std(axis=0, dtype=np.float64)
            mean[phase_columns] = phase_mean[phase_columns]
            std[phase_columns] = phase_std[phase_columns]
        std[std < 1e-6] = 1.
        baseline = np.zeros((9, V5_STEPS), dtype=np.float32)
        baseline_counts = np.zeros_like(baseline, dtype=np.int64)
        codes = _joint_codes(raw)
        if residual_baseline:
            fallback = float(np.median(y[mask]))
            pooled = np.array([np.median(y[:, t][mask[:, t]]) if mask[:, t].any()
                               else fallback for t in range(V5_STEPS)], dtype=np.float32)
            baseline[:] = pooled
            for group in range(9):
                group_mask = mask & (codes[:, None] == group)
                for step in range(V5_STEPS):
                    selected = group_mask[:, step]
                    baseline_counts[group, step] = int(selected.sum())
                    if selected.any():
                        baseline[group, step] = np.median(y[selected, step])
        residual = y-baseline[codes]
        scored = residual[mask]
        q25, q75 = np.quantile(scored, [.25, .75])
        return dict(input_shape=(V5_STEPS, n_features), feature_names=names,
            x_mean=mean, x_std=std, scaler_phase=scaler_phase,
            inspiration_scale_columns=phase_columns, target_center=float(np.median(scored)),
            target_scale=max(float(q75-q25), 1.), residual_baseline=bool(residual_baseline),
            baseline_profile=baseline, baseline_counts=baseline_counts,
            category_mappings=dict(R=list(V5_R_VALUES), C=list(V5_C_VALUES)),
            fitted_training_breaths=int(len(raw)), fitted_inspiratory_points=int(mask.sum()))


    def _validated_preprocessor(preprocess):
        prep = copy.deepcopy(preprocess)
        shape = tuple(prep['input_shape'])
        if len(shape) != 2 or shape[0] != V5_STEPS or shape[1] < 2:
            raise ValueError('Invalid checkpoint input shape')
        prep['input_shape'] = shape
        for key in ('x_mean', 'x_std'):
            prep[key] = np.array(prep[key], dtype=np.float32, copy=True)
            if prep[key].shape != (shape[1],) or not np.isfinite(prep[key]).all():
                raise ValueError('Invalid checkpoint input scaler')
        if (prep['x_std'] <= 0).any() or not np.isfinite(prep['target_center']):
            raise ValueError('Invalid input scale or target center')
        if not np.isfinite(prep['target_scale']) or prep['target_scale'] <= 0:
            raise ValueError('Invalid target scale')
        prep['baseline_profile'] = np.array(prep['baseline_profile'], dtype=np.float32, copy=True)
        if prep['baseline_profile'].shape != (9, V5_STEPS) or not np.isfinite(prep['baseline_profile']).all():
            raise ValueError('Invalid baseline profile')
        expected = dict(R=list(V5_R_VALUES), C=list(V5_C_VALUES))
        if prep['category_mappings'] != expected:
            raise ValueError('Checkpoint category mapping differs')
        return prep


    def transform_inputs(x, preprocess):
        raw, prep = _raw_x(x), _validated_preprocessor(preprocess)
        if raw.shape[1:] != prep['input_shape']:
            raise ValueError('Raw input shape differs from fitted preprocessor')
        standardized = (raw-prep['x_mean'])/prep['x_std']
        codes = np.broadcast_to(_joint_codes(raw)[:, None, None], (*raw.shape[:2], 1))
        transformed = np.ascontiguousarray(np.concatenate((standardized, codes), axis=-1), dtype=np.float32)
        if not np.isfinite(transformed).all():
            raise ValueError('Nonfinite standardized input')
        return transformed


    def _scaled_targets(raw, y, prep):
        return np.ascontiguousarray((y-prep['baseline_profile'][_joint_codes(raw)]
            -prep['target_center'])/prep['target_scale'], dtype=np.float32)


    def inspiratory_mae(y, prediction, u_out):
        y, prediction, out = np.asarray(y), np.asarray(prediction), np.asarray(u_out)
        if y.shape != prediction.shape or out.shape != y.shape or not (out == 0).any():
            raise ValueError('Metric shape or mask mismatch')
        if not np.isfinite(y).all() or not np.isfinite(prediction).all():
            raise ValueError('Nonfinite metric input')
        return float(np.abs(prediction.astype(np.float64)-y.astype(np.float64))[out == 0].mean())


    @torch.no_grad()
    def _predict_native(model, transformed, prep, device, batch_size):
        model.eval()
        output = np.empty(transformed.shape[:2], dtype=np.float32)
        for start in range(0, len(transformed), int(batch_size)):
            batch = torch.from_numpy(transformed[start:start+batch_size]).to(device)
            output[start:start+batch_size] = model(batch).cpu().numpy()
        codes = transformed[:, 0, -1].astype(np.int64)
        output = output*prep['target_scale']+prep['target_center']+prep['baseline_profile'][codes]
        if not np.isfinite(output).all():
            raise RuntimeError('Nonfinite native prediction')
        return output.astype(np.float32, copy=False)


    def load_checkpoint(checkpoint_path, device='cpu'):
        """Load trusted local v005 checkpoints. Older checkpoints need the old adapter."""
        payload = torch.load(Path(checkpoint_path), map_location='cpu', weights_only=False)
        if payload.get('format_version') != 5:
            raise ValueError('Expected a v005 checkpoint; use the matching older adapter')
        prep = _validated_preprocessor(payload['preprocess'])
        model = build_model(payload['model_spec'], prep['input_shape'][1]).to(_sequence_device(device))
        model.load_state_dict(payload['state_dict'], strict=True)
        model.eval()
        return model, prep, payload


    def predict_checkpoint(checkpoint_path, x, device='cpu', batch_size=512):
        model, prep, _ = load_checkpoint(checkpoint_path, device)
        return _predict_native(model, transform_inputs(x, prep), prep, _sequence_device(device), batch_size)

In [7]:
if RUN:
    # Schedules, checkpointing, and reproducible epoch primitives
    def _train_configuration(train_cfg):
        supplied = copy.deepcopy(dict(train_cfg))
        unknown = set(supplied)-set(V5_TRAIN_DEFAULTS)
        if unknown:
            raise ValueError(f'Unknown train fields: {sorted(unknown)}')
        cfg = dict(V5_TRAIN_DEFAULTS, **supplied)
        for key in ('batch_size', 'max_epochs', 'prediction_batch_size'):
            cfg[key] = int(cfg[key])
            if cfg[key] < 1:
                raise ValueError(f'{key} must be positive')
        cfg['patience'] = 0 if cfg['patience'] is None else int(cfg['patience'])
        cfg['scheduler_patience'], cfg['warmup_epochs'] = int(cfg['scheduler_patience']), int(cfg['warmup_epochs'])
        if min(cfg['patience'], cfg['scheduler_patience'], cfg['warmup_epochs']) < 0:
            raise ValueError('Patience and warmup counts must be nonnegative')
        if not (0 < cfg['lr'] and 0 <= cfg['min_lr'] <= cfg['lr'] and cfg['gradient_clip'] > 0):
            raise ValueError('Invalid learning-rate or clipping configuration')
        if cfg['max_seconds'] is not None and cfg['max_seconds'] <= 0:
            raise ValueError('max_seconds must be positive or None')
        if cfg['min_delta'] < 0 or cfg['weight_decay'] < 0 or not 0 < cfg['scheduler_factor'] < 1:
            raise ValueError('Invalid regularization or selection configuration')
        if cfg['scheduler_threshold'] < 0:
            raise ValueError('scheduler_threshold must be nonnegative')
        if cfg['scheduler'] not in ('plateau', 'cosine', 'constant', 'replay', 'fixed'):
            raise ValueError('scheduler must be plateau, cosine, constant, or replay/fixed')
        if cfg['warmup_epochs'] >= cfg['max_epochs']:
            raise ValueError('Warmup must end before the total epoch budget')
        if cfg['scheduler'] != 'cosine' and cfg['warmup_epochs']:
            raise ValueError('warmup_epochs is supported by the cosine schedule')
        if cfg['scheduler'] in ('replay', 'fixed'):
            if cfg['frozen_lr_schedule'] is None:
                raise ValueError('Replay scheduling requires frozen_lr_schedule')
            rates = [float(v) for v in cfg['frozen_lr_schedule']]
            if len(rates) != cfg['max_epochs'] or not all(np.isfinite(v) and v >= 0 for v in rates):
                raise ValueError('Frozen rates must contain max_epochs nonnegative finite values')
            cfg['frozen_lr_schedule'] = rates
        elif cfg['frozen_lr_schedule'] is not None:
            raise ValueError('frozen_lr_schedule applies only to replay/fixed scheduling')
        if cfg['optimizer'] not in ('Adam', 'AdamW'):
            raise ValueError('optimizer must be Adam or AdamW')
        if cfg['loss'] not in ('l1', 'smooth_l1') or cfg['native_beta'] <= 0:
            raise ValueError('loss must be l1 or smooth_l1 with native_beta > 0')
        if cfg['scaler_phase'] not in ('all', 'inspiration'):
            raise ValueError('Invalid scaler phase')
        for key, value in cfg.items():
            if isinstance(value, (float, np.floating)) and not np.isfinite(value):
                raise ValueError(f'Nonfinite configuration value: {key}')
        return cfg


    def deterministic_lr_schedule(train_cfg):
        """Return upfront LRs for cosine/constant/replay; plateau requires A feedback."""
        cfg = _train_configuration(train_cfg)
        if cfg['scheduler'] == 'plateau':
            return None
        if cfg['scheduler'] in ('replay', 'fixed'):
            return list(cfg['frozen_lr_schedule'])
        if cfg['scheduler'] == 'constant':
            return [float(cfg['lr'])]*cfg['max_epochs']
        warmup, total = cfg['warmup_epochs'], cfg['max_epochs']
        rates = []
        for epoch in range(1, total+1):
            if warmup and epoch <= warmup:
                rate = cfg['lr']*epoch/warmup
            else:
                denominator = max(total-warmup-1, 1)
                progress = (epoch-warmup-1)/denominator
                rate = cfg['min_lr']+.5*(cfg['lr']-cfg['min_lr'])*(1+math.cos(math.pi*progress))
            rates.append(float(rate))
        return rates


    def _cpu_copy(value):
        if torch.is_tensor(value):
            return value.detach().cpu().clone()
        if isinstance(value, dict):
            return {k: _cpu_copy(v) for k, v in value.items()}
        if isinstance(value, list):
            return [_cpu_copy(v) for v in value]
        if isinstance(value, tuple):
            return tuple(_cpu_copy(v) for v in value)
        return copy.deepcopy(value)


    def _rng_state(generator):
        return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
            cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            loader=generator.get_state())


    def _restore_rng(state, generator):
        random.setstate(state['python'])
        np.random.set_state(state['numpy'])
        torch.set_rng_state(state['torch'])
        if state['cuda'] is not None and torch.cuda.is_available():
            if len(state['cuda']) != torch.cuda.device_count():
                raise ValueError('CUDA device count changed across exact resumption')
            torch.cuda.set_rng_state_all(state['cuda'])
        generator.set_state(state['loader'])


    def _atomic_torch_save(payload, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name+'.tmp-'+uuid.uuid4().hex)
        try:
            torch.save(payload, temporary)
            os.replace(temporary, path)
        finally:
            if temporary.exists():
                temporary.unlink()


    def _file_sha256(path):
        result = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024*1024), b''):
                result.update(block)
        return result.hexdigest()


    def _array_sha256(values):
        array = np.ascontiguousarray(values)
        result = hashlib.sha256(str((array.shape, array.dtype.str)).encode())
        view = memoryview(array).cast('B')
        for start in range(0, len(view), 1024*1024):
            result.update(view[start:start+1024*1024])
        return result.hexdigest()


    def _json_safe(value):
        if isinstance(value, dict):
            return {k: _json_safe(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [_json_safe(v) for v in value]
        if isinstance(value, np.ndarray):
            return value.tolist()
        if isinstance(value, np.generic):
            return value.item()
        return value


    def _epoch_event(path, event):
        if path is not None:
            path = Path(path)
            path.parent.mkdir(parents=True, exist_ok=True)
            with path.open('a') as stream:
                stream.write(json.dumps(_json_safe(event), allow_nan=False)+'\n')


    def _last_path(checkpoint_path):
        path = Path(checkpoint_path)
        return path.with_name(path.stem+'.last'+path.suffix)


    def _fingerprint(raw, y, out, spec, cfg, seed, feature_names, mode, extra=None):
        return dict(mode=mode, model_spec=spec, train_cfg=cfg, seed=int(seed),
            feature_names=None if feature_names is None else list(feature_names),
            x_sha256=_array_sha256(raw), y_sha256=_array_sha256(y),
            out_sha256=_array_sha256(out), extra=extra)


    def _optimizer(model, cfg):
        optimizer_class = torch.optim.AdamW if cfg['optimizer'] == 'AdamW' else torch.optim.Adam
        return optimizer_class(model.parameters(), lr=float(cfg['lr']),
            betas=(.9, .999), eps=1e-8, weight_decay=float(cfg['weight_decay']))


    def _loader(raw, y, out, prep, cfg, seed, device):
        generator = torch.Generator().manual_seed(int(seed))
        transformed = transform_inputs(raw, prep)
        data = TensorDataset(torch.from_numpy(transformed), torch.from_numpy(_scaled_targets(raw, y, prep)),
            torch.from_numpy(np.ascontiguousarray(out == 0)))
        loader = DataLoader(data, batch_size=cfg['batch_size'], shuffle=True, num_workers=0,
            pin_memory=device.type == 'cuda', generator=generator)
        return loader, generator, transformed


    def _training_epoch(model, loader, optimizer, cfg, prep, device):
        model.train()
        total_error, total_count, optimizer_steps = 0., 0, 0
        for xb, yb, mask in loader:
            xb, yb, mask = (v.to(device, non_blocking=device.type == 'cuda') for v in (xb, yb, mask))
            if not bool(mask.any()):
                continue
            optimizer.zero_grad(set_to_none=True)
            difference = (model(xb)-yb).masked_select(mask)
            if cfg['loss'] == 'l1':
                loss = difference.abs().mean()
            else:
                beta = float(cfg['native_beta'])/prep['target_scale']
                loss = F.smooth_l1_loss(difference, torch.zeros_like(difference), beta=beta)
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), float(cfg['gradient_clip']))
            optimizer.step()
            optimizer_steps += 1
            total_error += float(difference.detach().abs().sum())*prep['target_scale']
            total_count += difference.numel()
        if not total_count:
            raise ValueError('Training epoch contained no scored points')
        return float(total_error/total_count), total_count, optimizer_steps


    def _checkpoint_payload(model, prep, spec, cfg, seed, fingerprint, optimizer, scheduler,
                            generator, epoch, history, completed=False, termination=None):
        return dict(format_version=5, model_spec=copy.deepcopy(spec), preprocess=_cpu_copy(prep),
            state_dict=_cpu_copy(model.state_dict()), optimizer_state=_cpu_copy(optimizer.state_dict()),
            scheduler_state=None if scheduler is None else copy.deepcopy(scheduler.state_dict()),
            rng_state=_cpu_copy(_rng_state(generator)), seed=int(seed), config=copy.deepcopy(cfg),
            fingerprint=copy.deepcopy(fingerprint), epoch=int(epoch), epochs=copy.deepcopy(history),
            optimizer_steps=sum(r['optimizer_steps'] for r in history),
            parameter_count=int(sum(p.numel() for p in model.parameters())),
            completed=bool(completed), termination=termination)


    def _prepare_run(raw, y, out, spec, cfg, seed, feature_names, checkpoint_path, resume,
                     fingerprint, device):
        checkpoint_path, last = Path(checkpoint_path), _last_path(checkpoint_path)
        if not resume and (checkpoint_path.exists() or last.exists()):
            raise FileExistsError('Use a fresh path or explicitly resume an existing stage')
        seed_sequence(seed)
        if resume:
            if not last.is_file():
                raise FileNotFoundError(f'Missing last-state checkpoint: {last}')
            saved = torch.load(last, map_location='cpu', weights_only=False)
            if saved.get('format_version') != 5 or saved['fingerprint'] != fingerprint:
                raise ValueError('Resume rejected: stage data, recipe, seed, or budget changed')
            prep = _validated_preprocessor(saved['preprocess'])
        else:
            saved = None
            prep = fit_preprocessor(raw, y, out, spec, feature_names,
                scaler_phase=cfg['scaler_phase'], residual_baseline=cfg['residual_baseline'])
        model = build_model(spec, raw.shape[-1]).to(device)
        optimizer = _optimizer(model, cfg)
        loader, generator, transformed = _loader(raw, y, out, prep, cfg, seed, device)
        return model, optimizer, loader, generator, transformed, prep, saved

In [8]:
if RUN:
    # A validation fitting with best state and exact boundary resumption
    def fit_validate(x_train, y_train, u_out_train, x_val, y_val, u_out_val,
                     model_spec, train_cfg, seed, checkpoint_path, epoch_log_path=None,
                     feature_names=None, train_diagnostic_breaths=2048, device='cuda',
                     resume=False, on_epoch=None):
        """Select on A only. Callers must supply disjoint whole-breath groups.

        `patience=0` or None disables early stopping. A time cap stops after the current
        epoch and returns completed=False; resume continues from the separate .last
        checkpoint, preserving Adam moments, scheduler, all RNGs and shuffle generator.
        The public checkpoint always contains the best A model; .last also retains the
        complete best snapshot so an interruption between file writes is recoverable.
        """
        began = time.monotonic()
        device, spec, cfg = _sequence_device(device), canonical_model_spec(model_spec), _train_configuration(train_cfg)
        raw, y, out = _training_arrays(x_train, y_train, u_out_train)
        val_raw, val_y, val_out = _training_arrays(x_val, y_val, u_out_val)
        if val_raw.shape[1:] != raw.shape[1:]:
            raise ValueError('Training and validation feature dimensions differ')
        if int(train_diagnostic_breaths) < 1:
            raise ValueError('train_diagnostic_breaths must be positive')
        extra = dict(val_x_sha256=_array_sha256(val_raw), val_y_sha256=_array_sha256(val_y),
            val_out_sha256=_array_sha256(val_out), train_diagnostic_breaths=int(train_diagnostic_breaths))
        fingerprint = _fingerprint(raw, y, out, spec, cfg, seed, feature_names, 'validation', extra)
        model, optimizer, loader, generator, train_z, prep, saved = _prepare_run(
            raw, y, out, spec, cfg, seed, feature_names, checkpoint_path, resume, fingerprint, device)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
            factor=cfg['scheduler_factor'], patience=cfg['scheduler_patience'],
            threshold=cfg['scheduler_threshold'], threshold_mode='rel', min_lr=cfg['min_lr']) \
            if cfg['scheduler'] == 'plateau' else None
        upfront_rates = deterministic_lr_schedule(cfg)
        val_z = transform_inputs(val_raw, prep)
        diagnostic_ids = np.sort(np.random.default_rng(20261110).choice(len(raw),
            size=min(int(train_diagnostic_breaths), len(raw)), replace=False))
        history, best_payload, bad_epochs, first_epoch, elapsed_before = [], None, 0, 1, 0.
        termination, completed = 'epoch_cap', False
        if saved is not None:
            model.load_state_dict(saved['state_dict'], strict=True)
            optimizer.load_state_dict(saved['optimizer_state'])
            if scheduler is not None:
                scheduler.load_state_dict(saved['scheduler_state'])
            _restore_rng(saved['rng_state'], generator)
            history, best_payload = copy.deepcopy(saved['epochs']), saved.get('best_payload')
            bad_epochs, first_epoch = saved.get('bad_epochs', 0), saved['epoch']+1
            elapsed_before = float(saved.get('elapsed_seconds', 0.))
            completed, termination = saved['completed'], saved['termination']
        else:
            initial = _checkpoint_payload(model, prep, spec, cfg, seed, fingerprint, optimizer,
                scheduler, generator, 0, [], completed=False, termination='initialized')
            initial.update(best_payload=None, bad_epochs=0, elapsed_seconds=0.)
            _atomic_torch_save(initial, _last_path(checkpoint_path))
        _epoch_event(epoch_log_path, dict(event='resume' if resume else 'start', first_epoch=first_epoch,
            model_spec=spec, train_cfg=cfg, deterministic_lr_schedule=upfront_rates,
            optimizer=cfg['optimizer'], weight_decay=cfg['weight_decay'],
            loss=cfg['loss'], scaled_beta=cfg['native_beta']/prep['target_scale'] if cfg['loss']=='smooth_l1' else None))
        if not completed:
            for epoch in range(first_epoch, cfg['max_epochs']+1):
                epoch_began = time.monotonic()
                if upfront_rates is not None:
                    for group in optimizer.param_groups:
                        group['lr'] = upfront_rates[epoch-1]
                lr_used = float(optimizer.param_groups[0]['lr'])
                online_mae, count, optimizer_steps = _training_epoch(model, loader, optimizer, cfg, prep, device)
                validation_mae = inspiratory_mae(val_y,
                    _predict_native(model, val_z, prep, device, cfg['prediction_batch_size']), val_out)
                train_eval_mae = inspiratory_mae(y[diagnostic_ids],
                    _predict_native(model, train_z[diagnostic_ids], prep, device, cfg['prediction_batch_size']),
                    out[diagnostic_ids])
                best_mae = float('inf') if best_payload is None else best_payload['best_val_mae']
                improved = validation_mae < best_mae-float(cfg['min_delta'])
                bad_epochs = 0 if improved else bad_epochs+1
                if scheduler is not None:
                    scheduler.step(validation_mae)
                if upfront_rates is not None:
                    lr_next = upfront_rates[epoch] if epoch < len(upfront_rates) else None
                else:
                    lr_next = float(optimizer.param_groups[0]['lr'])
                elapsed = elapsed_before+time.monotonic()-began
                record = dict(event='epoch', epoch=epoch, lr_used=lr_used, lr_next=lr_next,
                    training_online_inspiratory_mae=online_mae, training_eval_inspiratory_mae=train_eval_mae,
                    training_diagnostic_breaths=len(diagnostic_ids), training_scored_points=count,
                    optimizer_steps=optimizer_steps,
                    cumulative_optimizer_steps=sum(r['optimizer_steps'] for r in history)+optimizer_steps,
                    validation_inspiratory_mae=validation_mae,
                    best_val_mae=validation_mae if improved else best_mae,
                    best_epoch=epoch if improved else best_payload['best_epoch'],
                    epoch_seconds=time.monotonic()-epoch_began, elapsed_seconds=elapsed)
                history.append(record)
                completed, termination = epoch == cfg['max_epochs'], 'epoch_cap'
                if cfg['patience'] and bad_epochs >= cfg['patience']:
                    completed, termination = True, 'early_stopping'
                if not completed and cfg['max_seconds'] is not None and time.monotonic()-began >= cfg['max_seconds']:
                    termination = 'time_cap'
                current = _checkpoint_payload(model, prep, spec, cfg, seed, fingerprint, optimizer,
                    scheduler, generator, epoch, history, completed, termination)
                current.update(bad_epochs=bad_epochs, elapsed_seconds=elapsed)
                if improved:
                    best_payload = copy.deepcopy(current)
                    best_payload.update(best_epoch=epoch, best_val_mae=validation_mae,
                        training_eval_mae=train_eval_mae,
                        selected_lr_schedule=[r['lr_used'] for r in history], validation_used=True)
                current['best_payload'] = best_payload
                # Last is authoritative for resumption, including its best snapshot.
                _atomic_torch_save(current, _last_path(checkpoint_path))
                if improved:
                    _atomic_torch_save(best_payload, checkpoint_path)
                _epoch_event(epoch_log_path, record)
                if on_epoch is not None:
                    on_epoch(copy.deepcopy(record))
                if completed or termination == 'time_cap':
                    break
        if best_payload is None:
            raise RuntimeError('No validation epoch has completed')
        model.load_state_dict(best_payload['state_dict'], strict=True)
        restored = inspiratory_mae(val_y,
            _predict_native(model, val_z, prep, device, cfg['prediction_batch_size']), val_out)
        if not np.isclose(restored, best_payload['best_val_mae'], rtol=1e-6, atol=1e-6):
            raise RuntimeError('Restored checkpoint metric differs')
        best_payload.update(executed_epochs=len(history), executed_lr_schedule=[r['lr_used'] for r in history],
            completed=bool(completed), termination=termination,
            stage_completed=bool(completed), stage_termination=termination)
        _atomic_torch_save(best_payload, checkpoint_path)
        report = dict(model_spec=spec, train_cfg=cfg, seed=int(seed), validation_mae=restored,
            best_epoch=best_payload['best_epoch'], selected_lr_schedule=best_payload['selected_lr_schedule'],
            deterministic_lr_schedule=upfront_rates, epochs=history, executed_epochs=len(history),
            executed_lr_schedule=[r['lr_used'] for r in history], completed=bool(completed), termination=termination,
            optimizer_steps=sum(r['optimizer_steps'] for r in history),
            training_eval_mae=best_payload['training_eval_mae'], training_diagnostic_breaths=len(diagnostic_ids),
            training_breaths=len(raw), validation_breaths=len(val_raw),
            scored_validation_points=int((val_out == 0).sum()),
            elapsed_seconds=elapsed_before+time.monotonic()-began,
            parameter_count=best_payload['parameter_count'], checkpoint_path=str(checkpoint_path),
            checkpoint_sha256=_file_sha256(checkpoint_path), last_checkpoint_path=str(_last_path(checkpoint_path)),
            preprocessing=_json_safe(prep), validation_used=True, best_checkpoint_restored=True)
        _epoch_event(epoch_log_path, dict(event='finished', completed=completed, termination=termination,
            best_epoch=report['best_epoch'], validation_mae=restored, checkpoint_sha256=report['checkpoint_sha256']))
        return _json_safe(report)

In [9]:
if RUN:
    # Frozen A-schedule fitting for B or all labeled breaths
    def fit_fixed(x_train, y_train, u_out_train, model_spec, train_cfg, seed, lr_schedule,
                  checkpoint_path, epoch_log_path=None, feature_names=None, device='cuda',
                  resume=False, on_epoch=None):
        """Replay exactly one supplied LR per full epoch; accepts no validation arrays.

        Fits fresh training-only scalers/profiles. Saves each completed epoch and all
        optimizer/RNG states. No scheduler or early stopping consults B/all-data labels.
        A time cap returns completed=False and can be resumed at the next epoch.
        """
        began = time.monotonic()
        device, spec, cfg = _sequence_device(device), canonical_model_spec(model_spec), _train_configuration(train_cfg)
        raw, y, out = _training_arrays(x_train, y_train, u_out_train)
        rates = [float(rate) for rate in lr_schedule]
        if not rates or not all(np.isfinite(rate) and rate >= 0 for rate in rates):
            raise ValueError('A nonempty frozen sequence of nonnegative finite LRs is required')
        fingerprint = _fingerprint(raw, y, out, spec, cfg, seed, feature_names, 'fixed',
            dict(lr_schedule=rates))
        model, optimizer, loader, generator, _, prep, saved = _prepare_run(
            raw, y, out, spec, cfg, seed, feature_names, checkpoint_path, resume, fingerprint, device)
        history, first_epoch, elapsed_before = [], 1, 0.
        completed, termination = False, 'fixed_A_budget'
        if saved is not None:
            model.load_state_dict(saved['state_dict'], strict=True)
            optimizer.load_state_dict(saved['optimizer_state'])
            _restore_rng(saved['rng_state'], generator)
            history, first_epoch = copy.deepcopy(saved['epochs']), saved['epoch']+1
            elapsed_before = float(saved.get('elapsed_seconds', 0.))
            completed, termination = saved['completed'], saved['termination']
        else:
            initial = _checkpoint_payload(model, prep, spec, cfg, seed, fingerprint, optimizer,
                None, generator, 0, [], completed=False, termination='initialized')
            initial.update(elapsed_seconds=0., validation_used=False, selected_lr_schedule=rates)
            _atomic_torch_save(initial, _last_path(checkpoint_path))
        _epoch_event(epoch_log_path, dict(event='resume' if resume else 'start', first_epoch=first_epoch,
            model_spec=spec, train_cfg=cfg, frozen_lr_schedule=rates, validation_used=False))
        if not completed:
            for epoch in range(first_epoch, len(rates)+1):
                epoch_began = time.monotonic()
                for group in optimizer.param_groups:
                    group['lr'] = rates[epoch-1]
                online_mae, count, optimizer_steps = _training_epoch(model, loader, optimizer, cfg, prep, device)
                elapsed = elapsed_before+time.monotonic()-began
                record = dict(event='epoch', epoch=epoch, lr_used=rates[epoch-1],
                    lr_next=rates[epoch] if epoch < len(rates) else None,
                    training_online_inspiratory_mae=online_mae, training_scored_points=count,
                    optimizer_steps=optimizer_steps,
                    cumulative_optimizer_steps=sum(r['optimizer_steps'] for r in history)+optimizer_steps,
                    epoch_seconds=time.monotonic()-epoch_began, elapsed_seconds=elapsed, validation_used=False)
                history.append(record)
                completed, termination = epoch == len(rates), 'fixed_A_budget'
                if not completed and cfg['max_seconds'] is not None and time.monotonic()-began >= cfg['max_seconds']:
                    termination = 'time_cap'
                payload = _checkpoint_payload(model, prep, spec, cfg, seed, fingerprint, optimizer,
                    None, generator, epoch, history, completed, termination)
                payload.update(elapsed_seconds=elapsed, validation_used=False,
                    selected_lr_schedule=rates, completed_epochs=epoch)
                _atomic_torch_save(payload, _last_path(checkpoint_path))
                _atomic_torch_save(payload, checkpoint_path)
                _epoch_event(epoch_log_path, record)
                if on_epoch is not None:
                    on_epoch(copy.deepcopy(record))
                if completed or termination == 'time_cap':
                    break
        # A completed resume may have only the authoritative .last file after interruption.
        if saved is not None and first_epoch > len(rates):
            _atomic_torch_save(saved, checkpoint_path)
        report = dict(model_spec=spec, train_cfg=cfg, seed=int(seed), epochs=history,
            selected_lr_schedule=rates, executed_lr_schedule=[r['lr_used'] for r in history],
            executed_epochs=len(history), completed_epochs=len(history), completed=bool(completed),
            optimizer_steps=sum(r['optimizer_steps'] for r in history),
            termination=termination, validation_used=False, training_breaths=len(raw),
            parameter_count=int(sum(p.numel() for p in model.parameters())),
            elapsed_seconds=elapsed_before+time.monotonic()-began, checkpoint_path=str(checkpoint_path),
            checkpoint_sha256=_file_sha256(checkpoint_path), last_checkpoint_path=str(_last_path(checkpoint_path)),
            preprocessing=_json_safe(prep))
        _epoch_event(epoch_log_path, dict(event='finished', completed=completed, termination=termination,
            checkpoint_sha256=report['checkpoint_sha256']))
        return _json_safe(report)

In [10]:
def extend_completed_validation_checkpoint(source_last,source_best,target_checkpoint,expected_old_fingerprint,new_cfg):
    """Explicit budget extension; preserve model/Adam/RNG/shuffle/preprocessing state.

    Only max_epochs and frozen_lr_schedule may differ. New rates must replay every
    original rate, then remain at the original floor. Never silently relabel a
    checkpoint trained on different data/features/loss/seeds/optimizer settings.
    """
    source_last,source_best,target_checkpoint=map(Path,[source_last,source_best,target_checkpoint])
    target_last=_last_path(target_checkpoint)
    if target_last.exists() or target_checkpoint.exists():raise FileExistsError('Extension target is not fresh')
    saved=torch.load(source_last,map_location='cpu',weights_only=False)
    public=torch.load(source_best,map_location='cpu',weights_only=False)
    old_cfg=_train_configuration(saved['config']);new_cfg=_train_configuration(new_cfg)
    assert saved['format_version']==5 and saved['fingerprint']==expected_old_fingerprint
    assert public['fingerprint']==expected_old_fingerprint and public['model_spec']==saved['model_spec']
    assert saved['completed'] and saved['termination']=='epoch_cap'
    old_epochs=old_cfg['max_epochs'];assert saved['epoch']==old_epochs==len(saved['epochs'])
    assert new_cfg['max_epochs']>old_epochs and old_cfg['scheduler']==new_cfg['scheduler']=='replay'
    assert saved['scheduler_state'] is None and saved['best_payload'] is not None
    changed={key for key in old_cfg if old_cfg[key]!=new_cfg[key]};assert changed=={'max_epochs','frozen_lr_schedule'}
    old_rates=deterministic_lr_schedule(old_cfg);new_rates=deterministic_lr_schedule(new_cfg)
    assert new_rates[:old_epochs]==old_rates==[e['lr_used'] for e in saved['epochs']]
    assert new_rates[old_epochs:]==[old_cfg['min_lr']]*(new_cfg['max_epochs']-old_epochs)
    assert public['best_epoch']==saved['best_payload']['best_epoch']
    assert public['best_val_mae']==saved['best_payload']['best_val_mae']
    for key,value in public['state_dict'].items():assert torch.equal(value,saved['best_payload']['state_dict'][key])
    assert signature(_json_safe(public['preprocess']))==signature(_json_safe(saved['preprocess']))
    fingerprint=copy.deepcopy(expected_old_fingerprint);fingerprint['train_cfg']=new_cfg
    migrated=copy.deepcopy(saved);migrated['config']=new_cfg;migrated['fingerprint']=fingerprint
    migrated['completed']=False;migrated['termination']='epoch_cap'
    migrated['epochs'][-1]['lr_next']=new_rates[old_epochs]
    best=migrated['best_payload'];best['config']=copy.deepcopy(new_cfg);best['fingerprint']=copy.deepcopy(fingerprint)
    best['completed']=False
    if best['epoch']==old_epochs:best['epochs'][-1]['lr_next']=new_rates[old_epochs]
    public=copy.deepcopy(public);public['config']=copy.deepcopy(new_cfg);public['fingerprint']=copy.deepcopy(fingerprint)
    public['completed']=False;public['stage_completed']=False;public['stage_termination']='extended_prefix'
    provenance=dict(source_last=str(source_last),source_last_sha256=sha(source_last),source_best=str(source_best),source_best_sha256=sha(source_best),old_epochs=old_epochs,new_epochs=new_cfg['max_epochs'],only_configuration_changes=sorted(changed),model_optimizer_rng_preprocessing_unchanged=True,history_boundary_lr_next_updated=True)
    migrated['extension_provenance']=provenance;public['extension_provenance']=provenance
    _atomic_torch_save(migrated,target_last);_atomic_torch_save(public,target_checkpoint)
    return provenance

def extension_resume_proof(device='cpu'):
    import tempfile
    if device=='cpu':torch.set_num_threads(1)
    rng=np.random.default_rng(20261126);raw=rng.normal(size=(12,80,6)).astype(np.float32)
    raw[:,:,0]=np.resize(np.array([5.,20.,50.],dtype=np.float32),12)[:,None]
    raw[:,:,1]=np.resize(np.array([10.,20.,50.,20.],dtype=np.float32),12)[:,None]
    y=(.02*raw[:,:,0]+.01*raw[:,:,1]+raw[:,:,2]*.2+np.linspace(0,1,80)[None,:]).astype(np.float32)
    out=np.ones((12,80),dtype=np.int8);out[:,:32]=0;names=['R','C','a','b','c','d']
    train=np.arange(8);val=np.arange(8,12);spec=dict(family='lstm',hidden=4,layers=2,bidirectional=True,dropout=.15,head_width=4)
    old_rates=[.001,.001,.0005,.0001];new_rates=old_rates+[.0001,.0001]
    cfg4=_train_configuration(dict(lr=.001,batch_size=4,max_epochs=4,patience=0,max_seconds=None,min_lr=.0001,scheduler='replay',frozen_lr_schedule=old_rates))
    cfg6=dict(cfg4,max_epochs=6,frozen_lr_schedule=new_rates)
    with tempfile.TemporaryDirectory(prefix='pa3-v006-extension-proof-') as temporary:
        directory=Path(temporary);direct=directory/'direct.pt';prefix=directory/'prefix.pt';extended=directory/'extended.pt'
        args=(raw[train],y[train],out[train],raw[val],y[val],out[val],spec)
        common=dict(feature_names=names,train_diagnostic_breaths=4,device=device)
        full=fit_validate(*args,cfg6,20261126,direct,**common)
        fit_validate(*args,cfg4,20261126,prefix,**common)
        old=torch.load(_last_path(prefix),map_location='cpu',weights_only=False)
        provenance=extend_completed_validation_checkpoint(_last_path(prefix),prefix,extended,old['fingerprint'],cfg6)
        resumed=fit_validate(*args,cfg6,20261126,extended,resume=True,**common)
        a=torch.load(_last_path(direct),map_location='cpu',weights_only=False);b=torch.load(_last_path(extended),map_location='cpu',weights_only=False)
        checks=[0]
        def same(x,z):
            if torch.is_tensor(x):assert torch.equal(x,z)
            elif isinstance(x,np.ndarray):assert np.array_equal(x,z)
            elif isinstance(x,dict):
                assert x.keys()==z.keys()
                for key in x:same(x[key],z[key])
            elif isinstance(x,(list,tuple)):
                assert len(x)==len(z)
                for p,q in zip(x,z):same(p,q)
            else:assert x==z
            checks[0]+=1
        for key in ['state_dict','optimizer_state','rng_state']:same(a[key],b[key])
        assert full['best_epoch']==resumed['best_epoch'] and full['validation_mae']==resumed['validation_mae']
        for x,z in zip(a['epochs'],b['epochs']):
            for key in ['epoch','lr_used','lr_next','training_online_inspiratory_mae','training_eval_inspiratory_mae','validation_inspiratory_mae','best_val_mae','best_epoch','optimizer_steps','cumulative_optimizer_steps']:same(x[key],z[key])
        pa=predict_checkpoint(direct,raw[val],device=device);pb=predict_checkpoint(extended,raw[val],device=device);same(pa,pb)
        # Reject any simultaneous change in loss, architecture/data or rate prefix.
        bad=dict(cfg6,loss='smooth_l1')
        try:extend_completed_validation_checkpoint(_last_path(prefix),prefix,directory/'bad.pt',old['fingerprint'],bad)
        except AssertionError:checks[0]+=1
        else:raise AssertionError('Changed loss was accepted')
        return dict(status='passed',device=str(device),exact_state_checks=checks[0],direct_6_equals_prefix4_plus2=True,changed_loss_rejected=True,assignment_data_used=False)


In [11]:
if RUN:
    assert FINAL_GATE.is_file(),'Current C/final pipeline must finish before pilot training'
    gate=json.loads(FINAL_GATE.read_text());assert gate['status']=='complete' and gate['training_breaths']==67905
    assert sha(Path(gate['submission_file']))==gate['submission_sha256']
    OUT.mkdir(parents=True,exist_ok=True)
    INNER_LOCK=(OUT/'.run.lock').open('a+');fcntl.flock(INNER_LOCK,fcntl.LOCK_EX|fcntl.LOCK_NB)
    proof=extension_resume_proof('cuda');write_json(OUT/'course_extension_proof.json',proof)
    quality=json.loads((PROJECT/'quality_audit_v002.json').read_text());assert sha(PROJECT/'data/train.csv')==quality['files']['train.csv']['sha256']
    base_design=json.loads((BASE/'design.json').read_text());assert code_hash(PROJECT/'iterative_improvements_v005.ipynb')==base_design['source_code_sha256']
    BASELINE=json.loads((BASE/'record_A_capacity_bilstm128.json').read_text());assert BASELINE['training_breaths']==61114 and BASELINE['feature_mode']=='combined' and BASELINE['executed_epochs']==200
    assert sha(Path(BASELINE['checkpoint_path']))==BASELINE['checkpoint_sha256']
    oldcfg=_train_configuration(BASELINE['recipe']['train_cfg']);oldspec=canonical_model_spec(BASELINE['recipe']['model_spec'])
    assert oldcfg['scheduler']=='replay' and oldcfg['max_epochs']==200 and oldcfg['frozen_lr_schedule']==BASELINE['executed_lr_schedule']
    duration_cfg=dict(oldcfg,max_epochs=300,frozen_lr_schedule=oldcfg['frozen_lr_schedule']+[oldcfg['min_lr']]*100)
    cosine_cfg=dict(oldcfg,max_epochs=300,scheduler='cosine',warmup_epochs=5,frozen_lr_schedule=None)
    head_spec=dict(oldspec,rc_heads=True)
    ARMS=dict(H1_duration300=dict(model_spec=oldspec,train_cfg=duration_cfg),H1_slowcosine300=dict(model_spec=oldspec,train_cfg=cosine_cfg),H2_RC_heads200=dict(model_spec=head_spec,train_cfg=oldcfg))
    changes={k for k in oldspec if oldspec[k]!=head_spec[k]};assert changes=={'rc_heads'}
    raw_frame=pd.read_csv(PROJECT/'data/train.csv',dtype=dict(id='int64',breath_id='int32',R='int8',C='int8',time_step='float32',u_in='float32',u_out='int8',pressure='float32'))
    breath_ids=_validate_feature_frame_v005(raw_frame);assert len(breath_ids)==67905
    Y=raw_frame.pressure.to_numpy(copy=True).reshape(-1,80);UOUT=raw_frame.u_out.to_numpy(copy=True).reshape(-1,80);ROW_IDS=raw_frame.id.to_numpy(copy=True).reshape(-1,80);RC=raw_frame[['R','C']].to_numpy(copy=True)[::80]
    frame=raw_frame[['id','breath_id','R','C','time_step','u_in','u_out']].copy();del raw_frame;gc.collect()
    index=pd.Index(breath_ids)
    with np.load(BASE/'split_breath_ids.npz',allow_pickle=False) as saved:GROUPS={key:np.sort(index.get_indexer(saved[key])) for key in ['A_fit','A_validation']}
    assert len(GROUPS['A_fit'])==61114 and len(GROUPS['A_validation'])==6791
    assert all((v>=0).all() and len(np.unique(v))==len(v) for v in GROUPS.values())
    assert np.array_equal(np.sort(np.r_[GROUPS['A_fit'],GROUPS['A_validation']]),np.arange(67905))
    group_hashes={key:hashlib.sha256(breath_ids[v].tobytes()).hexdigest() for key,v in GROUPS.items()}
    assert all(group_hashes[key]==base_design['group_hashes'][key] for key in GROUPS)
    design=dict(plan=PLAN,arms=ARMS,source_code_sha256=code_hash(PROJECT/'approved_pilots_v006.ipynb'),train_sha256=quality['files']['train.csv']['sha256'],group_hashes=group_hashes,feature_mode_dimensions=base_design['feature_mode_dimensions'],baseline_record_sha256=sha(BASE/'record_A_capacity_bilstm128.json'),baseline_checkpoint_sha256=BASELINE['checkpoint_sha256'],final_gate_sha256=sha(FINAL_GATE))
    design=json.loads(json.dumps(design))
    if (OUT/'design.json').exists():assert RESUME and json.loads((OUT/'design.json').read_text())==design
    else:write_json(OUT/'design.json',design)
    mask=UOUT[GROUPS['A_validation']]==0;row_ids=ROW_IDS[GROUPS['A_validation']][mask]
    if (OUT/'A_validation_row_ids.npy').exists():assert np.array_equal(np.load(OUT/'A_validation_row_ids.npy'),row_ids)
    else:np.save(OUT/'A_validation_row_ids.npy',row_ids)
    assert np.array_equal(row_ids,np.load(BASE/'A_validation_row_ids.npy'))
    FEATURE_MODE=FEATURE_X=FEATURE_NAMES=None;RECORDS={}


In [12]:
def events(event_name,**values):
    with (OUT/'events.jsonl').open('a') as f:f.write(json.dumps(dict(timestamp_utc=datetime.now(timezone.utc).isoformat(),event=event_name,**values))+'\n')
def get_features(mode):
    global FEATURE_MODE,FEATURE_X,FEATURE_NAMES
    if FEATURE_MODE!=mode:
        FEATURE_X=None;gc.collect()
        FEATURE_X,FEATURE_NAMES,b=make_sequence_features(frame,mode)
        assert np.array_equal(b,breath_ids)
        assert FEATURE_X.shape==(67905,80,design['feature_mode_dimensions'][mode])
        FEATURE_MODE=mode
    return FEATURE_X,FEATURE_NAMES
def refresh_results():
    rows=[]
    for p in OUT.glob('record_A_*.json'):
        r=json.loads(p.read_text());rows.append({k:r.get(k) for k in ['name','phase','feature_mode','validation_mae','training_eval_mae','best_epoch','executed_epochs','parameter_count','elapsed_seconds','seed']})
    if rows:pd.DataFrame(rows).sort_values(['validation_mae','name']).to_csv(OUT/'results.csv',index=False)
def fit_A(name,phase,mode,spec,cfg,seed=SEED):
    spec=canonical_model_spec(spec);cfg=_train_configuration(cfg)
    recipe=json.loads(json.dumps(dict(feature_mode=mode,model_spec=spec,train_cfg=cfg,seed=seed)))
    identity=signature(dict(recipe=recipe,design=signature(design),split='A'))
    path=OUT/('record_A_'+name+'.json')
    if path.exists():
        r=json.loads(path.read_text());assert r['identity']==identity
        assert sha(r['checkpoint_path'])==r['checkpoint_sha256']
        p=np.load(r['prediction_file']);assert hashlib.sha256(p.tobytes()).hexdigest()==r['prediction_sha256']
        report=json.loads(Path(r['report_file']).read_text())
        assert report['completed'] and report['executed_epochs']==cfg['max_epochs']
        val=GROUPS['A_validation'];mask=UOUT[val]==0;rows=ROW_IDS[val][mask]
        assert p.shape==rows.shape and np.isfinite(p).all()
        assert np.array_equal(np.load(OUT/'A_validation_row_ids.npy'),rows)
        assert np.isclose(np.abs(p-Y[val][mask]).mean(),r['validation_mae'],rtol=0,atol=1e-10)
        assert r['fit_group_sha256']==design['group_hashes']['A_fit']
        RECORDS[name]=r;return r
    x,names=get_features(mode);fit=GROUPS['A_fit'];val=GROUPS['A_validation'];mask=UOUT[val]==0
    checkpoint=OUT/('A_'+name+'.pt');last=checkpoint.with_name(checkpoint.stem+'.last'+checkpoint.suffix)
    epochlog=OUT/('A_'+name+'_epochs.jsonl')
    resume=last.exists();assert not resume or RESUME
    events('stage_started',name=name,phase=phase,recipe=recipe,resume=resume)
    report=fit_validate(x[fit],Y[fit],UOUT[fit],x[val],Y[val],UOUT[val],spec,cfg,seed,
        checkpoint,epoch_log_path=epochlog,feature_names=names,device='cuda',resume=resume)
    write_json(OUT/('report_A_'+name+'.json'),report)
    if not report['completed']:
        events('stage_incomplete',name=name,executed_epochs=report['executed_epochs'])
        raise RuntimeError('Stage saved for resumption before its full budget completed')
    prediction=predict_checkpoint(checkpoint,x[val],device='cuda',batch_size=cfg['prediction_batch_size'])[mask].astype(np.float64)
    truth=Y[val][mask];score=float(np.abs(prediction-truth).mean())
    assert np.isclose(score,report['validation_mae'],atol=1e-6,rtol=1e-6)
    rows=ROW_IDS[val][mask]
    if (OUT/'A_validation_row_ids.npy').exists():assert np.array_equal(np.load(OUT/'A_validation_row_ids.npy'),rows)
    else:np.save(OUT/'A_validation_row_ids.npy',rows)
    np.save(OUT/('A_'+name+'_predictions.npy'),prediction)
    r=dict(name=name,phase=phase,feature_mode=mode,recipe=recipe,identity=identity,seed=seed,
        validation_mae=score,training_eval_mae=report['training_eval_mae'],
        checkpoint_path=str(checkpoint),checkpoint_sha256=sha(checkpoint),
        prediction_file=str(OUT/('A_'+name+'_predictions.npy')),
        prediction_sha256=hashlib.sha256(prediction.tobytes()).hexdigest(),
        feature_names=names,training_breaths=len(fit),fit_group_sha256=design['group_hashes']['A_fit'],
        best_epoch=report['best_epoch'],selected_lr_schedule=report['selected_lr_schedule'],
        executed_lr_schedule=report['executed_lr_schedule'],executed_epochs=report['executed_epochs'],
        parameter_count=report['parameter_count'],elapsed_seconds=report['elapsed_seconds'],
        report_file=str(OUT/('report_A_'+name+'.json')))
    rc=np.repeat(RC[val],80,axis=0)[mask.ravel()]
    table=pd.DataFrame(dict(R=rc[:,0],C=rc[:,1],absolute_error=np.abs(prediction-truth),signed_error=prediction-truth))
    table.groupby(['R','C']).agg(points=('absolute_error','size'),mae=('absolute_error','mean'),total_absolute_error=('absolute_error','sum'),bias=('signed_error','mean')).to_csv(OUT/('A_'+name+'_by_rc.csv'))
    write_json(path,r);RECORDS[name]=r;refresh_results()
    events('stage_completed',name=name,validation_mae=score,best_epoch=r['best_epoch'],seconds=r['elapsed_seconds'])
    gc.collect();torch.cuda.empty_cache();return r
def best_record(records):return min(records,key=lambda r:(r['validation_mae'],r['name']))

In [13]:
if RUN:
    x,names=get_features('combined');fit=GROUPS['A_fit'];val=GROUPS['A_validation']
    newpath=OUT/'A_H1_duration300.pt'
    if not _last_path(newpath).exists() and not (OUT/'record_A_H1_duration300.json').exists():
        oldfp=_fingerprint(x[fit],Y[fit],UOUT[fit],oldspec,oldcfg,SEED,names,'validation',dict(val_x_sha256=_array_sha256(x[val]),val_y_sha256=_array_sha256(Y[val]),val_out_sha256=_array_sha256(UOUT[val]),train_diagnostic_breaths=2048))
        provenance=extend_completed_validation_checkpoint(_last_path(Path(BASELINE['checkpoint_path'])),Path(BASELINE['checkpoint_path']),newpath,oldfp,duration_cfg)
        write_json(OUT/'H1_extension_provenance.json',provenance)
    fit_A('H1_duration300','H1_duration','combined',oldspec,duration_cfg)


In [14]:
if RUN:
    fit_A('H1_slowcosine300','H1_schedule','combined',oldspec,cosine_cfg)

In [15]:
if RUN:
    fit_A('H2_RC_heads200','H2_heads','combined',head_spec,oldcfg)

In [16]:
if RUN:
    val=GROUPS['A_validation'];mask=UOUT[val]==0;target=Y[val][mask].astype(np.float64)
    deep=np.load(BASE/'A_capacity_deep_bilstm_predictions.npy').astype(np.float64)
    original=np.load(BASE/'A_capacity_bilstm128_predictions.npy').astype(np.float64)
    reference=.5*deep+.5*original;reference_mae=float(np.abs(reference-target).mean())
    results=[]
    for name,r in RECORDS.items():
        prediction=np.load(r['prediction_file']).astype(np.float64);pair=.5*deep+.5*prediction
        epochs=json.loads(Path(r['report_file']).read_text())['epochs'];added=epochs[200:] if name=='H1_duration300' else epochs
        results.append(dict(name=name,individual_A_mae=r['validation_mae'],fixed_equal_pair_A_mae=float(np.abs(pair-target).mean()),individual_improvement_vs_compact=BASELINE['validation_mae']-r['validation_mae'],pair_improvement_vs_reference=reference_mae-float(np.abs(pair-target).mean()),new_epoch_training_seconds=sum(e['epoch_seconds'] for e in added),best_epoch=r['best_epoch']))
    pd.DataFrame(results).sort_values('fixed_equal_pair_A_mae').to_csv(OUT/'pilot_comparison.csv',index=False)
    write_json(OUT/'pilot_summary.json',dict(status='complete',reference_A_pair_mae=reference_mae,reference_A_individual_mae=BASELINE['validation_mae'],results=results,C_confirmation_performed=False,final_all_fit_performed=False,submission_performed=False,H3_H4_started=False))
    events('approved_H1_H2_A_screen_complete',results=results)
    fcntl.flock(INNER_LOCK,fcntl.LOCK_UN);INNER_LOCK.close()
if not RUN and (OUT/'pilot_summary.json').exists():print((OUT/'pilot_summary.json').read_text())
